# File: **energy_balance.csv**

In [ ]:
######################################## Parameters

### Run
name = 'case_heating_1'
prefix = ''

In [ ]:
##### Import packages
import os
import sys
import fnmatch
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
warnings.filterwarnings('ignore', category=UserWarning)

Load file and show its content.

In [ ]:
df = xp.load_file_csv(
    params,
    filename='energy_balance.csv',
    location='results',
    prefix=prefix,
    name=name,
    folder='csvs',
    skiprows=4,
    header=None,
    names=['component', 'carrier', 'bus_carrier', 'value'],
)

df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])
df.head()

## bar summary

Show in a bar plot the balance per item and group.

In [ ]:
#################### Parameters

### Threshold to ignore items
threshold = 1e6 # MWh (and tCO2 for co2 before scaling)

### Unit conversion
unit_scale = 1e6  # MWh -> TWh and tCO2 -> MtCO2

### Token for carrier plot. If True, all group plots use the same carriers/order and fill missing values with 0
show_all = True

### Group definition
dic_group = {
    'electricity': ['AC', 'DC', 'low voltage', 'battery', 'EV battery', 'home battery'],
    'H2': ['H2'],
    'gas': ['gas'],
    'heat': ['urban central heat', 'urban decentral heat', 'rural heat',
             'urban central water tanks', 'urban decentral water tanks', 'rural water tanks',
             'urban central water pits'],
    'co2': ['co2', 'co2 stored', 'co2 sequestered', 'co2 emitted'],
}

dic_units = {
    'electricity': 'TWh',
    'H2': 'TWh',
    'gas': 'TWh',
    'heat': 'TWh',
    'co2': 'MtCO2',
}

# Keys are dataframe columns. Each list gives preferred values in the desired order.
# Remaining carriers are appended sorted by descending value.
preferred_order_dic = {
    #'component': ['Generator', 'Link', 'Load'],
    'carrier': ['onwind', 'solar rooftop',
                'urban_central*', 'urban_decentral*', 'rural*'],
    'group': ['electricity', 'H2', 'gas', 'heat', 'co2'],
}


#################### Operations

df_filtered = df.copy()
df_filtered['group'] = df_filtered['bus_carrier'].apply(
    lambda x: next((g for g, carriers in dic_group.items() if x in carriers), 'other')
)

# Row-level threshold filter (warn on dropped rows)
_row_mask = df_filtered['value'].abs() >= threshold
_dropped_rows = df_filtered.loc[~_row_mask]
df_filtered = df_filtered.loc[_row_mask]
if not _dropped_rows.empty:
    print(f"[Row filter] {len(_dropped_rows)} row(s) dropped (|value| < {threshold:g}):")
    for _, r in _dropped_rows.iterrows():
        print(f"  - {r['component']:>10} | {r['carrier']:<32} | {r['bus_carrier']:<24} | {r['value']:+.3e}")
    print()


#################### Plot helpers

def _wrap_two_lines(label, max_len=20):
    label = str(label)
    if len(label) <= max_len:
        return label
    midpoint = len(label) // 2
    left_space = label.rfind(' ', 0, midpoint + 1)
    right_space = label.find(' ', midpoint)
    if left_space != -1:
        split_at = left_space
    elif right_space != -1:
        split_at = right_space
    else:
        split_at = midpoint
    return label[:split_at].rstrip() + '\n' + label[split_at:].lstrip()


def _ordered_carriers(df_scope, values_by_carrier):
    # Sort everything by values_by_carrier (the values being plotted).
    preferred = []
    for col, preferred_values in preferred_order_dic.items():
        if col not in df_scope.columns:
            continue
        for pref in preferred_values:
            if '*' in pref:
                candidates = [c for c in values_by_carrier.index if fnmatch.fnmatch(c, pref)]
            else:
                in_scope = df_scope.loc[df_scope[col] == pref, 'carrier'].unique()
                candidates = [c for c in in_scope if c in values_by_carrier.index]
            if not candidates:
                continue
            for c in values_by_carrier.loc[candidates].sort_values(ascending=False).index:
                if c not in preferred:
                    preferred.append(c)

    remaining = [c for c in values_by_carrier.index if c not in preferred]
    remaining_sorted = values_by_carrier.loc[remaining].sort_values(ascending=False).index.tolist()
    return preferred + remaining_sorted


def _plot_group(group, plot_series, display_labels, y_range=None):
    labels_wrapped = [_wrap_two_lines(lbl, max_len=20) for lbl in display_labels]
    positive_sum = float(plot_series[plot_series > 0].sum())
    unit = dic_units.get(group, 'value')

    fig_width = max(8, 0.8 * len(plot_series))
    fig, ax = plt.subplots(figsize=(fig_width, 5))

    colors = ['tab:blue' if v >= 0 else 'tab:red' for v in plot_series.values]
    ax.bar(labels_wrapped, plot_series.values, color=colors, edgecolor='black', linewidth=0.6)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f"{group} ({positive_sum:.2f} {unit})")
    ax.set_xlabel('carrier')
    ax.set_ylabel(unit)
    ax.tick_params(axis='x', rotation=60)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.set_axisbelow(True)

    if y_range is not None:
        y_min, y_max = y_range
        if y_min == y_max:
            delta = 1.0 if y_min == 0 else abs(y_min) * 0.1
            y_min -= delta
            y_max += delta
        else:
            pad = (y_max - y_min) * 0.05
            y_min -= pad
            y_max += pad
        ax.set_ylim(y_min, y_max)

    plt.tight_layout()


#################### Plot

group_items = list(df_filtered.groupby('group', sort=False))
if not group_items:
    print('No data to plot after grouping.')
else:
    group_df_map = dict(group_items)
    groups_present = list(group_df_map.keys())

    # Apply preferred group order if provided, then append remaining groups
    preferred_groups = [g for g in preferred_order_dic.get('group', []) if g in groups_present]
    group_order = preferred_groups + [g for g in groups_present if g not in preferred_groups]

    # Merge rows by carrier, apply threshold to the merged sum, detect "balance" carriers, and log warnings.
    values_by_group = {}
    balance_by_group = {}
    _merge_log = {}     # group -> [(carrier, n_rows, merged_value)]
    _drop_log = {}      # group -> [(carrier, merged_value)]
    for g in group_order:
        grouped = group_df_map[g].groupby('carrier')['value']
        merged = grouped.sum()
        counts = grouped.size()
        mask = merged.abs() >= threshold
        values_by_group[g] = merged[mask]
        balance_by_group[g] = set(counts[mask & (counts > 1)].index)

        merged_carriers = counts[counts > 1]
        if not merged_carriers.empty:
            _merge_log[g] = [(c, int(cnt), float(merged[c])) for c, cnt in merged_carriers.items()]
        dropped_carriers = merged[~mask]
        if not dropped_carriers.empty:
            _drop_log[g] = [(c, float(v)) for c, v in dropped_carriers.items()]

    if _merge_log:
        print(f"[Merge] Carriers obtained by summing multiple rows:")
        for g, items in _merge_log.items():
            print(f"  {g}:")
            for c, cnt, v in items:
                print(f"    - {c}: {cnt} rows -> sum = {v:+.3e}")
        print()

    if _drop_log:
        print(f"[Merge filter] Carriers dropped after merge (|sum| < {threshold:g}):")
        for g, items in _drop_log.items():
            print(f"  {g}:")
            for c, v in items:
                print(f"    - {c}: {v:+.3e}")
        print()

    group_order = [g for g in group_order if not values_by_group[g].empty]

    if not group_order:
        print('No data to plot after applying threshold to merged carriers.')
    else:
        # Convert to plotting units per group: divide by 1e6 for TWh and MtCO2
        values_plot_by_group = {}
        for g in group_order:
            unit = dic_units.get(g, 'value')
            scale = unit_scale if unit in ('TWh', 'MtCO2') else 1.0
            values_plot_by_group[g] = values_by_group[g] / scale

        if show_all:
            # Shared carrier order using the full plotted scope
            global_values = pd.concat(values_by_group.values()).groupby(level=0).sum()
            scope_df = df_filtered.loc[df_filtered['group'].isin(group_order)]
            master_order = _ordered_carriers(scope_df, global_values)

            # A carrier is labelled " balance" if it was merged in ANY group
            balance_anywhere = set().union(*balance_by_group.values()) if balance_by_group else set()
            master_display_labels = [
                f"{c} balance" if c in balance_anywhere else c
                for c in master_order
            ]

            # Reindexed series + shared y-range per unit
            plot_series_by_group = {
                g: values_plot_by_group[g].reindex(master_order, fill_value=0.0)
                for g in group_order
            }
            unit_ranges = {}
            for g in group_order:
                unit = dic_units.get(g, 'value')
                s = plot_series_by_group[g]
                low = min(0.0, float(s.min()))
                high = max(0.0, float(s.max()))
                rng = unit_ranges.setdefault(unit, [low, high])
                rng[0] = min(rng[0], low)
                rng[1] = max(rng[1], high)

            for g in group_order:
                _plot_group(
                    g,
                    plot_series_by_group[g],
                    master_display_labels,
                    y_range=unit_ranges[dic_units.get(g, 'value')],
                )
        else:
            for g in group_order:
                values_plot = values_plot_by_group[g]
                order = _ordered_carriers(group_df_map[g], values_plot)
                plot_series = values_plot.loc[order]

                balance_set = balance_by_group[g]
                display_labels = [
                    f"{c} balance" if c in balance_set else c
                    for c in plot_series.index
                ]
                _plot_group(g, plot_series, display_labels)


In [ ]:
df_filtered

## Waterfall chart

The chart is built in three pieces, all configured through the dictionary `dic` defined right below:

- **`init_generation`** &mdash; first stacked column. Each entry contributes one segment to the *Generation* bar.
- **`intermediate_processes`** &mdash; one waterfall bar per entry. The bar height is the signed sum of the rows selected; positive values raise the running total, negative values lower it. Dashed grey segments connect consecutive levels.
- **`final_loads`** &mdash; last stacked column (values are sign-flipped so consumption is shown as positive). The running total should match this bar's height when the accounting is closed.

Each entry maps a legend name to a *selector* that picks rows from `energy_balance.csv`:

- `(component, carrier, bus_carrier)` &mdash; one specific row.
- A list of such tuples &mdash; sum of the rows it contains (useful to group losses spanning several buses).

Edit `dic` to add, remove or rearrange concepts; the plot adapts automatically.

In [ ]:
#################### Parameters

### Dic to arrange plot
dic = {
    'init_generation': {
        'Onshore wind': ('Generator', 'onwind', 'AC'),
        'Solar rooftop': ('Generator', 'solar rooftop', 'low voltage'),
    },
    'intermediate_processes': {
        'H2 electrolysis': ('Link', 'H2 Electrolysis', 'AC'),
        'H2 turbine': ('Link', 'H2 turbine', 'AC'),        
        'losses (transmission-distribution grid)': [
            ('Link', 'electricity distribution grid', 'AC'),
            ('Link', 'electricity distribution grid', 'low voltage'),
        ],        
        'losses (BEV charger)': [
            ('Link', 'BEV charger', 'low voltage'),
            ('Link', 'BEV charger', 'EV battery'),
        ],
        'V2G': ('Link', 'V2G', 'EV battery'),
    },
    'final_loads': {
        'Land transport EV': ('Load', 'land transport EV', 'EV battery'),
    }
}


### Units
unit_scale = 1e6
unit_label = 'TWh'

### Font sizes
label_fontsize = 14
tick_fontsize = 14
legend_fontsize = 14

In [ ]:
######################################## Helpers

def _resolve_selector(selector):
    """Normalize a selector to a list of (component, carrier, bus_carrier) tuples."""
    # Single-row selector like ('Generator', 'onwind', 'AC') -> wrap in list
    if isinstance(selector, tuple) and selector and isinstance(selector[0], str):
        selector = [selector]
    normalized = []
    for row in selector:
        if len(row) != 3:
            raise ValueError(f"Each row selector must have 3 items (component, carrier, bus_carrier), got {len(row)}")
        normalized.append(tuple(row))
    return normalized


def _selector_value(grouped, selector):
    """Sum values matching a selector against the pre-aggregated table."""
    total = 0.0
    for component, carrier, bus_carrier in _resolve_selector(selector):
        total += grouped.get((component, carrier, bus_carrier), 0.0)
    return total


def _series_from_dict(grouped, d, sign=1):
    return pd.Series(
        {name: sign * _selector_value(grouped, sel) for name, sel in d.items()},
        dtype=float,
    )


######################################## Pre-aggregate (one pass over the dataframe)

grouped = df_filtered.groupby(['component', 'carrier', 'bus_carrier'])['value'].sum()


######################################## Build stages

init_series = _series_from_dict(grouped, dic.get('init_generation', {})) / unit_scale
final_series = _series_from_dict(grouped, dic.get('final_loads', {}), sign=-1) / unit_scale
intermediate_deltas = pd.Series(
    {name: _selector_value(grouped, sel) / unit_scale
     for name, sel in dic.get('intermediate_processes', {}).items()},
    dtype=float,
)

stages = []
if not init_series.empty:
    stages.append({'kind': 'stacked', 'label': 'Generation', 'series': init_series})
for name, delta in intermediate_deltas.items():
    stages.append({'kind': 'delta', 'label': name, 'delta': delta})
if not final_series.empty:
    stages.append({'kind': 'stacked', 'label': 'Loads', 'series': final_series})

if not stages:
    raise ValueError("No data to plot. Check 'dic' configuration.")


######################################## Colors (tech_colors from pypsa-spain config/plotting.default.yaml)

plotting_cfg = xp.load_file_yaml(
    params,
    filename='plotting.default.yaml',
    location='config',
)
tech_colors = plotting_cfg['plotting']['tech_colors']
color_fallback = '#999999'


def _color_for(selector):
    """Resolve a color from tech_colors using the carrier of the first selector row."""
    carrier = _resolve_selector(selector)[0][1]
    return tech_colors.get(carrier, color_fallback)


colors_by_name = {
    name: _color_for(selector)
    for section in ('init_generation', 'intermediate_processes', 'final_loads')
    for name, selector in dic.get(section, {}).items()
}


######################################## Plot helpers

def _plot_stacked(ax, x, series, colors):
    pos_bottom = 0.0
    neg_bottom = 0.0
    for name, value in series.items():
        bottom = pos_bottom if value >= 0 else neg_bottom
        ax.bar(
            x, value, bottom=bottom,
            color=colors[name],
            edgecolor='black', linewidth=0.6,
            label=name,
        )
        if value >= 0:
            pos_bottom += value
        else:
            neg_bottom += value


def _plot_level_link(ax, x, level):
    ax.plot(
        [x - 1, x], [level, level],
        color='gray', linestyle='--', linewidth=0.8,
    )


######################################## Figure

fig, ax = plt.subplots(figsize=(12, 6))

# Reference magnitude to space the numeric labels above/below the delta bars
all_magnitudes = (
    list(init_series.abs().values)
    + list(intermediate_deltas.abs().values)
    + list(final_series.abs().values)
)
label_offset = 0.015 * max(all_magnitudes + [1.0])

current_level = 0.0

for idx, stage in enumerate(stages):

    if stage['kind'] == 'stacked':
        _plot_stacked(ax, idx, stage['series'], colors_by_name)

        if stage['label'] == 'Generation':
            current_level = stage['series'].sum()
        elif idx > 0:
            # Connect waterfall to the final stacked bar
            _plot_level_link(ax, idx, current_level)

    else:  # 'delta'
        delta = stage['delta']
        level_before = current_level
        level_after = current_level + delta

        bar_bottom = min(level_before, level_after)
        bar_height = abs(delta)

        ax.bar(
            idx, bar_height, bottom=bar_bottom,
            color=colors_by_name[stage['label']],
            edgecolor='black', linewidth=0.6,
            label='_nolegend_',
        )

        if idx > 0:
            _plot_level_link(ax, idx, level_before)

        if delta >= 0:
            text_y, text_va = bar_bottom + bar_height + label_offset, 'bottom'
        else:
            text_y, text_va = bar_bottom - label_offset, 'top'
        ax.text(
            idx, text_y, f"{delta:+.2f}",
            ha='center', va=text_va,
            fontsize=max(tick_fontsize - 1, 9),
        )

        current_level = level_after


######################################## Cosmetic

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(range(len(stages)))
ax.set_xticklabels([s['label'] for s in stages], rotation=20, ha='right', fontsize=tick_fontsize)
ax.set_ylabel(f"Energy [{unit_label}]", fontsize=label_fontsize)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)

# Deduplicated legend (skip intermediate bars)
handles, labels = ax.get_legend_handles_labels()
seen = set()
legend_handles, legend_labels = [], []
for h, l in zip(handles, labels):
    if l == '_nolegend_' or l in seen:
        continue
    seen.add(l)
    legend_handles.append(h)
    legend_labels.append(l)

ax.legend(
    legend_handles, legend_labels,
    loc='upper left', bbox_to_anchor=(1.01, 1),
    frameon=False, fontsize=legend_fontsize,
)

plt.tight_layout()
plt.show()
